In [ ]:
import os
import sys

# Repo root whether the notebook cwd is <repo>/tacotron2 or <repo>/
_cwd = os.getcwd()
if os.path.basename(_cwd.rstrip("/\\")) == "tacotron2":
    _REPO_ROOT = os.path.dirname(_cwd)
else:
    _REPO_ROOT = _cwd
for _p in (_REPO_ROOT, os.path.join(_REPO_ROOT, "tacotron2")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import torch
import matplotlib.pyplot as plt
from IPython.display import Audio

from commons.dataset import AudioMelConversions
from commons.hyperparams import Tacotron2Config
from model import Tacotron2
from tokenizer import Tokenizer

# Notebook-style checkpoint: model_state_dict + config (tries common layouts)
_CKPT_CANDIDATES = [
    os.path.join(_REPO_ROOT, "checkpoints_taco2", "tacotron2_epoch_0045.pth"),
    os.path.join(_REPO_ROOT, "tacotron2-wavernn", "checkpoints_taco2", "tacotron2_epoch_0045.pth"),
]
CHECKPOINT_PATH = next(
    (os.path.normpath(p) for p in _CKPT_CANDIDATES if os.path.isfile(p)), None
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRIFFIN_LIM_ITERS = 60


def _load_checkpoint(path: str, map_location):
    try:
        ck = torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        ck = torch.load(path, map_location=map_location)
    if isinstance(ck, dict) and "model_state_dict" in ck:
        return ck.get("config"), ck["model_state_dict"]
    return None, ck


if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "No checkpoint found. Tried:\n"
        + "\n".join(f"  - {os.path.normpath(p)}" for p in _CKPT_CANDIDATES)
        + "\nSet CHECKPOINT_PATH to your .pth file explicitly."
    )

saved_config, state_dict = _load_checkpoint(CHECKPOINT_PATH, DEVICE)
config = saved_config if saved_config is not None else Tacotron2Config()
model = Tacotron2(config).to(DEVICE)
model.load_state_dict(state_dict, strict=True)
model.eval()

tokenizer = Tokenizer()
a2m = AudioMelConversions(
    num_mels=config.num_mels,
    sampling_rate=config.sample_rate,
    n_fft=config.n_fft,
    window_size=config.win_length,
    hop_size=config.hop_length,
    fmin=config.fmin,
    fmax=config.fmax,
    min_db=config.min_db,
    max_scaled_abs=config.max_scaled_abs,
)
print(f"Loaded {CHECKPOINT_PATH} on {DEVICE}")


def inference(text, griffin_lim_iters=GRIFFIN_LIM_ITERS, max_decode_steps=1000):
    """Synthesize with Tacotron2 + Griffin–Lim; show mel, attention, and play audio."""
    print(f"Input text: {text!r}")
    tokens = tokenizer.encode(text).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        mel_post, alignments = model.inference(tokens, max_decode_steps=max_decode_steps)

    fig, axes = plt.subplots(2, 1, figsize=(6, 7))
    im0 = axes[0].imshow(
        mel_post[0].T.cpu().numpy(), aspect="auto", origin="lower", interpolation="none"
    )
    axes[0].set_title("Predicted mel (post-net)")
    axes[0].set_ylabel("Mel bin")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(
        alignments[0].T.cpu().numpy(), aspect="auto", origin="lower", interpolation="none"
    )
    axes[1].set_title("Attention alignment")
    axes[1].set_xlabel("Decoder time")
    axes[1].set_ylabel("Encoder position")
    fig.colorbar(im1, ax=axes[1])
    plt.tight_layout()
    plt.show()

    audio = a2m.mel2audio(
        mel_post[0].T.cpu(), do_denorm=True, griffin_lim_iters=griffin_lim_iters
    )
    display(Audio(audio, rate=config.sample_rate))
    return mel_post, alignments, audio


In [ ]:
inference(
    "شكرا جزيلا لاستماعكم"
)
